In [0]:
import pandas as pd
import mlflow
import mlflow.sklearn

from pyspark.sql import SparkSession

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier


spark = SparkSession.builder.getOrCreate()


mlflow.set_experiment(
    "/Users/danniel.lisardo10@gmail.com/lead_scoring_experiment"
)


df = spark.read.table("medallion.gold.lead_scoring_dataset")

pdf = df.toPandas()



y = pdf["convertido"]

X = pdf.drop(
    columns=[
        "convertido",
        "lead_id",
        "processing_timestamp"
    ]
)



num_cols = [
    "idade",
    "tempo_preenchimento_seg",
    "hora_dia",
    "dia_semana",
    "mes",
    "taxa_conv_funil",
    "taxa_conv_vendedor",
    "taxa_conv_vendedor_funil",
    "volume_funil_dia"
]

cat_cols = [
    "funil",
    "canal_origem",
    "escolaridade",
    "turno_contato",
    "turno_preferido",
    "renda_faixa"
]

text_col = "objetivo_texto"


preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("text", TfidfVectorizer(max_features=50), text_col)
    ]
)


model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42
)


pipeline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", model)
    ]
)



split_date = "2025-07-01"

train = pdf[pdf["data_hora"] < split_date]
test = pdf[pdf["data_hora"] >= split_date]

X_train = train.drop(
    columns=["convertido", "lead_id", "processing_timestamp"]
)

y_train = train["convertido"]

X_test = test.drop(
    columns=["convertido", "lead_id", "processing_timestamp"]
)

y_test = test["convertido"]



with mlflow.start_run():

    pipeline.fit(X_train, y_train)


 

    probs = pipeline.predict_proba(X_test)[:, 1]

    test = test.copy()
    test["score"] = probs

    k = 100

    top_k = test.sort_values("score", ascending=False).head(k)

    precision_k = top_k["convertido"].mean()

    print("Precision@100:", precision_k)


    mlflow.log_metric("precision_at_100", precision_k)


    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 200)



    mlflow.sklearn.log_model(
        pipeline,
        name="lead_scoring_model",
        registered_model_name="lead_scoring_model",
        input_example=X_train.head(5)
    )


Precision@100: 0.36


🔗 View Logged Model at: https://dbc-42d71d80-ef5b.cloud.databricks.com/ml/experiments/176631697807569/models/m-4d73072057e64280870d2aa0996bb134?o=7474644078994030
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: Us

Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.lead_scoring_model': https://dbc-42d71d80-ef5b.cloud.databricks.com/explore/data/models/workspace/default/lead_scoring_model/version/1?o=7474644078994030
